In [ ]:
# Standard library imports =====================================
import numpy as np
import math
import sys
import random
from scipy.optimize import lsq_linear

# Custom function imports ======================================
from rtelem import minelem
from sremove import rstates

poparray = np.sqrt(1/3)*np.array([1,1,1])
# Rules ========================================================
# i, j, k, l, m, and n are all reserved for integer incrementing
print('poparray:',poparray)


#	Not super efficient, could put sooner, but here is the rank reduction 
#	to working with only populated electronic states -> add a tolerance to pass
#	from namd-main.py

vstates = []
dimH = len(poparray)
zpop = 1.0e-6
i = 0
while i < dimH:
    if (poparray[i] >= zpop):
        vstates.append(i)
    i = i + 1
pass

rank = len(vstates)
print('rank:',rank)
#	print 'odotrho'
#	print odotrho
#	print 'w'
#	print w

#	sys.exit()
# constructing vectorized target density matrix
# This target is rho**(d) in the original python manuscript
# However it is organized into a column vector of the diagonal and
# lower triangular elements so it can be used as the target in a 
# library linear least squares optimization

eseg = np.zeros((rank,rank))
nblock = 0

vtarget=[1, 0.05, 0.3, 1, 0.9, 1]
#print('vtarget_matrix:',vtarget_matrix)
#print('precollapseentropy:',precollapseentropy)
iter = 0        # Used to track what column is sent to minelem
bcore = []      # block coordinates for fastest decaying element
listbank = []
nzthresh = 1.0e-6
eseg = ([1, 0.05, 0.3], [0.05, 1, 0.9], [0.3, 0.9, 1])
dgscale = 1.0e6
p, q, leave = minelem(eseg,rank,nzthresh)

bcore.append(p)
bcore.append(q)

#       print 'bcore'
#       print bcore

#       print 'pre-algorithm check before individual element stuff is made'
#       sys.exit()

while leave == 'no':

            # Finding the largest possible coherent sub-block
    i = 0
    bstates = []
    while i < rank:         # starting by adding all possible states
        bstates.append(i)
        i = i + 1
    pass

            # checking for zeroes in eseg such that blocks can be removed from the coherent block
    icstates = []   # list of states with zero elemnts in rows or columns shared by the
                            # coherence indicated by bcore

    if (nblock > 0):
        icstates = rstates(rank,eseg,bcore,nzthresh)    # function identifying the icoherent states
    pass

#               print 'icstates', icstates

    i = 0
    if (nblock == 0):
        pass
    else:
#                       print('is this getting killed by poor loop navigation?')
        while i < len(icstates):
            j = -(i + 1)
            temp = bstates.pop(icstates[j])
            i = i + 1
        pass
    pass

    listbank.append(bstates[:])

#               print 'bstates for current block', bstates

            # projecting out the current block from eseg
    iter = len(bstates)
    i = 0
    temp4 = eseg[bcore[0]][bcore[1]]
    while i < iter:
        j = 0
        while j < iter:
            eseg[bstates[i]][bstates[j]] = eseg[bstates[i]][bstates[j]] - temp4
            j = j + 1
        pass
        i = i + 1
    pass

#               print 'updated eseg'
#               print eseg

    nblock = nblock + 1
    if (nblock > 500):
        print('nblock blew up')
        print('Pnc matrix')
#                        print(Pnc)
        print('current eseg')
        print(eseg)
        sys.exit()
    pass

    bcore = []      # block coordinates for fastest decaying elements

    p, q, leave = minelem(eseg,rank,nzthresh)

    bcore.append(p)
    bcore.append(q)

pass

i = 0
while i < rank:
    if (abs(poparray[vstates[i]]*eseg[i][i]) > nzthresh):
        listbank.append([])
        listbank[-1].append(i)
    pass
    i = i + 1
pass


#	print 'vstates', vstates
#	print 'listbank'
#	print listbank




#	print 'vtarget'
#	print vtarget

#	sys.exit()

# generating the vectorized set of coherent block density matrices
# for the TAB wave function collapse step
# These are determined using the current time step electronic populations,
# and therefore must be recomputed at each collapse step. (Unlike the list of 
# states populated in each block [listbank])

A = [] 	#stores the block density matrices in form of A[block-ID][element]
aindex = 0


i = 0
while i < len(listbank):
    k = 0
    A.append([])
    while k < rank:
        l = k
        while l < rank:
            if (k in listbank[i] and l in listbank[i]):
                m = 0
                popsum = 0.0
                while m < len(listbank[i]):
                    popsum = popsum + poparray[vstates[listbank[i][m]]]
                    m = m + 1
                pass
                if (popsum <= nzthresh):
                    aelem = 0.0
                else:
                    if (k == l):
                        aelem = dgscale*((poparray[vstates[k]]*poparray[vstates[l]])**(0.5))/popsum
                    else:
                        aelem = ((poparray[vstates[k]]*poparray[vstates[l]])**(0.5))/popsum
                    pass
                pass
            else:
                aelem = 0.0
            A[aindex].append(aelem)
            l = l + 1
        pass
        k = k + 1
    pass
    i = i + 1
    aindex = aindex + 1
pass

#	Test to see if there is linear dependence, and if not will do the real
#	linear least squares optimization
At = np.transpose(A)

#	ehrv = []
#	ehrv.append([])

#	i = 0
#	while i < len(A[-1]):
#		ehrv[0].append(A[-1][i])
#		i = i + 1
#	pass

#	ehrvt = np.transpose(ehrv)
#	ehrw = lsq_linear(ehrvt, vtarget, bounds=(0.0, 2.0), method='bvls')

#	if (abs(ehrw.x[0]-1.0) <= pehrptol):
#		i = 0
#		return poparray
#	pass


#	print 'A[dimH-1]'
#	print A[dimH-1]

# scipy linear least squares optimization
#	At = np.transpose(A)
optw = lsq_linear(At, vtarget, bounds=(0.0, 2.0), method='bvls')

# Error analysis of the linear least squares target wave function

#Convergence analysis
# Sort the weights in descending order
sorted_weights = np.sort(optw.x)[::-1]

# Initialize cumulative sum and term counter
cumulative_sum = 0.0
term_count = 0

# Calculate how many terms it takes to reach or exceed 0.90 cumulative weight
for weight in sorted_weights:
    cumulative_sum += weight



# Collapsing the wave function

i = 0
ptotal = 0.0
while i < len(optw.x):
    ptotal = ptotal + optw.x[i]
    i = i + 1
pass

#	print 'optw'
#	print optw.x

check2 = random.random()
check = check2*ptotal

track = 0
psum = 0.0
i = 0
j = 0
while j < 2:
    psum = psum + optw.x[i]
    if (check <= psum):
        j = 3
        track = i
    else:
        i = i + 1
    pass
pass

# track is the collapsed into density matrix according to A
# constructing npop from the density matrix
print('poparray:',poparray)
if (track == 0):
    print('track == 0')
    return poparray, precollapseentropy.real, cumulative_sum
pass
k = 0
index2 = 0
while k < rank:
    l = k
    while l < rank:
        if (k == l):
            npop[vstates[k]] = A[track][index2]/dgscale
        index2 = index2 + 1
        l = l + 1
    k = k + 1
pass

#	print 'vectorized target'
#	print A[track]
return npop, precollapseentropy.real, cumulative_sum